[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nattaponm/Teach_stormTracks_KKN_May2024_HID_doppler/blob/main/prep/P4_package_release.ipynb)


# P4 — จัดแพ็กข้อมูล: repo (วันหลัก + ตารางทั้งเดือน) และ Zenodo (กริดรายวันทั้งเดือน)

**สำหรับผู้ดูแลข้อมูล** — ขั้นสุดท้าย

1. ตรวจไฟล์รายวันทั้งหมด (CAPPI, polar, labels) และคำนวณ SHA-256
2. เลือก **วันหลัก** ที่จะเก็บกริดไว้ใน repo (ไม่เกิน `repo_max_mb`) — นิสิตรันบท 02–06 ได้ทันทีโดยไม่ต้องดาวน์โหลด
3. เขียน `data/derived/day_index.csv` และ `config/data_sources.yaml` ให้ `utils/data.py` รู้ว่าไฟล์ไหนอยู่ที่ไหน
4. รวม labels เป็น zip ละ Δt (Zenodo จำกัด 100 ไฟล์/record) แล้ว (เลือกได้) อัปโหลดขึ้น Zenodo แบบ **draft** ด้วย token ใน Colab Secrets (`ZENODO_TOKEN`) แล้วไปตรวจและกด Publish เองบนเว็บ

> ข้อมูลที่เผยแพร่เป็นกริด Cartesian 1 km ที่ประมวลผลแล้วเท่านั้น ไม่มีไฟล์ UF หรือข้อมูลในพิกัด polar

In [13]:
# @title ⚙️ เตรียมสภาพแวดล้อม (รันก่อนเสมอ)
import os, sys, subprocess
REPO = "Teach_stormTracks_KKN_May2024_HID_doppler"
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir(f"/content/{REPO}"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", f"https://github.com/nattaponm/{REPO}.git", f"/content/{REPO}"], check=True)
    else:
        subprocess.run(["git", "-C", f"/content/{REPO}", "pull", "-q"])
    os.chdir(f"/content/{REPO}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) in ("notebooks", "prep"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("โฟลเดอร์ทำงาน:", os.getcwd(), "| CPU", os.cpu_count(), "core")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
โฟลเดอร์ทำงาน: /content/Teach_stormTracks_KKN_May2024_HID_doppler | CPU 8 core


In [14]:
import json, time, shutil, warnings
from pathlib import Path
from functools import partial
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mproc
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
warnings.filterwarnings("ignore")

from utils.config import load_settings, repo_path
from utils import maps as M
s = load_settings()
M.setup_style()
OUT_ROOT = Path(os.environ.get("KKN_OUT_ROOT", s["radar_data"]["drive_derived_root"]))
N_WORKERS = os.cpu_count()
CTX = mproc.get_context("fork")

# ดึงผลของขั้นก่อนหน้าจาก Drive (Colab แยกเครื่องต่อ notebook — ดู utils/sync.py)
from utils import sync
sync.pull(OUT_ROOT)
s = load_settings()      # อ่านใหม่หลัง pull (พิกัดเรดาร์จริงจาก P0)

import hashlib, yaml
from utils import data_io
DER, DAYS_DIR = repo_path("data", "derived"), repo_path("data", "days")
KINDS = {"cappi_15min": OUT_ROOT / "cappi", "polar_15min": OUT_ROOT / "polar"}
KINDS.update({f"labels_{t}": OUT_ROOT / "tracks" for t in s["tracking"]["tags"]})

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

rows = []
for kind, folder in KINDS.items():
    for p in sorted(folder.glob(f"KKN_*_{kind}.nc")):
        rows.append({"day": pd.Timestamp(p.name[4:12]), "kind": kind, "file": p.name, "path": str(p),
                     "mb": round(p.stat().st_size / 1e6, 3), "sha256": sha256(p)})
files = pd.DataFrame(rows)
print(f"{len(files)} ไฟล์ รวม {files['mb'].sum():.0f} MB")
files.pivot_table(index="kind", values="mb", aggfunc=["count", "sum"]).round(1)

sync.pull: /content/drive/MyDrive/000_research/08Storm_NorthEast_FF69/0data/1radar/KKN_derived_v1/repo_mirror → /content/Teach_stormTracks_KKN_May2024_HID_doppler
  คัดลอก 1 ไฟล์ (0.0 MB): data/derived/manifest.json
151 ไฟล์ รวม 415 MB


,count,sum
,mb,mb
kind,,
cappi_15min,31,123.9
labels_05min_linZ,30,11.9
labels_10min_linZ,30,6.2
labels_15min,30,4.4
polar_15min,30,268.6


## 1. เลือกวันหลักสำหรับ repo

ค่าตั้งต้น: 3 วันที่มีคะแนนความเสียหายในรัศมีสูงสุด (จากบท 01) + 1 วันในช่วงต้นฤดูฝนที่มีเส้นทาง ≥ 30 นาทีมากที่สุด (เป็นกลุ่มเปรียบเทียบ) ตัดออกทีละวันถ้าเกิน `repo_max_mb`

In [15]:
key = s["release"]["key_days"]
if key is None:
    rp = repo_path("data", "reports", "days_may2024.csv")
    if not rp.exists():
        raise FileNotFoundError("ไม่พบ data/reports/days_may2024.csv — รันบท 01 โดยตั้ง ADMIN_SYNC = True (เก็บผลเข้า Drive) "
                                "หรือ commit data/reports ก่อน หรือกำหนด release.key_days ใน config/settings.yaml เอง")
    rep_days = pd.read_csv(rp, parse_dates=["date"])
    key = list(rep_days.sort_values("D_in_range", ascending=False)["date"].head(3))
    T15 = pd.read_csv(DER / "tracks_15min.csv", parse_dates=["day"])
    rainy = T15[(T15["day"].dt.day >= 21) & (T15["lifetime_min"] >= 30)].groupby("day").size()
    if len(rainy):
        key.append(rainy.idxmax())
key = [pd.Timestamp(k) for k in key if pd.Timestamp(k) in set(files["day"])]
size = files[files["day"].isin(key)].groupby("day")["mb"].sum()
while size.sum() > s["release"]["repo_max_mb"] and len(size) > 1:
    size = size.iloc[:-1]
key = list(size.index)
print("วันหลัก:", [f"{k:%d %b}" for k in key], f"| {size.sum():.1f} MB")
for _, r in files[files["day"].isin(key)].iterrows():
    dst = DAYS_DIR / f"{r['day']:%Y%m%d}" / r["file"]
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(r["path"], dst)

วันหลัก: ['03 May', '05 May', '06 May', '22 May'] | 59.3 MB


## 2. ดัชนีวันและแหล่งข้อมูล

In [16]:
# Zenodo รับได้ไม่เกิน 100 ไฟล์ต่อ record → CAPPI/polar ขึ้นเป็นไฟล์รายวัน (61 ไฟล์)
# ส่วน labels (90 ไฟล์) รวมเป็น zip ละ Δt (3 ไฟล์) รวม 64 ไฟล์
ZENODO_RECORD_ID = "23333709"
ZENODO_DOI = "10.5281/zenodo.23333709"      # DOI ที่จองไว้ (ลงทะเบียนจริงเมื่อ Publish)
import zipfile
ARC_DIR = OUT_ROOT / "zenodo_archives"; ARC_DIR.mkdir(parents=True, exist_ok=True)

single = files[files["kind"].isin(["cappi_15min", "polar_15min"])].copy()
archives, arc_rows = {}, []
for tag in s["tracking"]["tags"]:
    part = files[files["kind"] == f"labels_{tag}"].sort_values("day")
    if part.empty:
        continue
    zname = f"KKN_May2024_labels_{tag}.zip"
    zpath = ARC_DIR / zname
    with zipfile.ZipFile(zpath, "w", compression=zipfile.ZIP_DEFLATED) as z:   # เขียนใหม่ทุกครั้ง (เนื้อหาเหมือนเดิม)
        for _, r in part.iterrows():
            zi = zipfile.ZipInfo(r["file"], date_time=(2024, 6, 1, 0, 0, 0))     # เวลาคงที่ → checksum ของ zip คงที่
            zi.compress_type = zipfile.ZIP_DEFLATED
            with open(r["path"], "rb") as f:
                z.writestr(zi, f.read())
    archives[zname] = {"mb": round(zpath.stat().st_size / 1e6, 3), "sha256": sha256(zpath),
                       "members": list(part["file"]), "member_sha256": dict(zip(part["file"], part["sha256"]))}
    arc_rows.append({"file": zname, "path": str(zpath), "mb": archives[zname]["mb"]})
    print(f"{zname}: {len(part)} วัน, {archives[zname]['mb']:.1f} MB")

upload = pd.concat([single[["file", "path", "mb"]], pd.DataFrame(arc_rows)], ignore_index=True)
print(f"ไฟล์ที่จะขึ้น Zenodo: {len(upload)} ไฟล์ (จำกัด 100) รวม {upload['mb'].sum():.0f} MB")
assert len(upload) <= 100

idx = files.groupby("day").agg(kinds=("kind", lambda k: ";".join(sorted(k))), mb=("mb", "sum")).reset_index()
idx["in_repo"] = idx["day"].isin(key)
idx["in_zenodo"] = True
idx.to_csv(DER / "day_index.csv", index=False)
src_path = repo_path("config", "data_sources.yaml")
zen = {"record_id": ZENODO_RECORD_ID, "doi": ZENODO_DOI, "sandbox": s["release"]["zenodo_sandbox"],
       "files": {r["file"]: {"mb": float(r["mb"]), "sha256": r["sha256"]} for _, r in single.iterrows()},
       "archives": archives}
with open(src_path, "w", encoding="utf-8") as f:
    yaml.safe_dump({"zenodo": zen}, f, allow_unicode=True, sort_keys=False)
manifest = {"created_utc": pd.Timestamp.utcnow().isoformat(), "n_files": len(files), "n_zenodo_files": len(upload),
            "total_mb": float(files["mb"].sum()), "zenodo_record": ZENODO_RECORD_ID, "doi": ZENODO_DOI,
            "key_days_in_repo": [f"{k:%Y-%m-%d}" for k in key],
            "derived_tables": sorted(p.name for p in DER.glob("*.*")),
            "processing": "prep/P0–P4 (Teach_stormTracks_KKN_May2024_HID_doppler)", "license": "CC BY 4.0",
            "radar_data_credit": "Thai Meteorological Department (KKN); UF files not distributed"}
json.dump(manifest, open(DER / "manifest.json", "w"), indent=1, ensure_ascii=False)
print("เขียน day_index.csv, config/data_sources.yaml (files + archives), manifest.json")

KKN_May2024_labels_15min.zip: 30 วัน, 1.9 MB
KKN_May2024_labels_10min_linZ.zip: 30 วัน, 2.7 MB
KKN_May2024_labels_05min_linZ.zip: 30 วัน, 5.4 MB
ไฟล์ที่จะขึ้น Zenodo: 64 ไฟล์ (จำกัด 100) รวม 402 MB
เขียน day_index.csv, config/data_sources.yaml (files + archives), manifest.json


## 3. อัปโหลด Zenodo (draft)

**ข้อจำกัด:** Zenodo รับได้ไม่เกิน 100 ไฟล์ต่อ record แต่ไฟล์รายวันมีทั้งหมด 151 ไฟล์ เซลล์ก่อนหน้าจึงรวม labels เป็น zip ละ Δt ทำให้เหลือ 64 ไฟล์ ส่วน `utils/data.py` จะแตก zip ให้นิสิตเองอัตโนมัติ

1. สร้าง token ที่ Zenodo → Applications → Personal access tokens โดยให้สิทธิ์ `deposit:write` และ `deposit:actions`
2. ใน Colab ไปที่แถบซ้าย 🔑 Secrets แล้วเพิ่ม `ZENODO_TOKEN`
3. ตั้ง `UPLOAD = True` แล้วรัน ไฟล์จะขึ้นเป็น **draft** ซึ่งยังไม่เผยแพร่
   - ไฟล์ที่มี md5 ตรงกับที่อยู่บน Zenodo แล้วจะถูกข้าม จึงรันซ้ำได้
   - ไฟล์ที่ไม่อยู่ในรายการ เช่น labels รายวันที่อัปโหลดไว้เดิม จะถูกแจ้งให้ทราบ และลบให้ถ้าตั้ง `DELETE_EXTRA = True`
4. ตรวจ metadata บนเว็บ แล้วกด Publish เอง
5. commit `config/data_sources.yaml` ซึ่งเซลล์ก่อนหน้าใส่ record id และ DOI ไว้แล้ว เพื่อให้นิสิตดาวน์โหลดวันอื่นได้

In [17]:
# ============================================================
# P4: Upload to an EXISTING Zenodo draft (≤ 100 files, retries, md5 skip)
# ============================================================
UPLOAD = True
DELETE_EXTRA = False        # True = ลบไฟล์บน draft ที่ไม่อยู่ในรายการ upload (เช่น labels รายวันที่ค้างอยู่)

if UPLOAD:
    import os, time, hashlib, requests
    from urllib.parse import quote
    from google.colab import userdata

    token = userdata.get("ZENODO_TOKEN")
    if not token:
        raise ValueError("ZENODO_TOKEN is missing from Colab Secrets.")
    base = "https://sandbox.zenodo.org" if s["release"]["zenodo_sandbox"] else "https://zenodo.org"
    S = requests.Session(); S.headers.update({"Authorization": f"Bearer {token}"})
    print("Zenodo:", base, "| record:", ZENODO_RECORD_ID)

    def call(method, url, tries=6, **kw):
        """HTTP with back-off on 5xx/429/timeouts; raises on 4xx. Never prints the token."""
        kw.setdefault("timeout", 120); r, err = None, ""
        for i in range(tries):
            try:
                r = S.request(method, url, **kw)
            except (requests.ConnectionError, requests.Timeout) as e:
                r, err = None, repr(e)[:120]
            if r is not None and r.ok:
                return r
            if r is not None and r.status_code < 500 and r.status_code != 429:
                raise RuntimeError(f"{method} {url.replace(base, '')} -> {r.status_code}: {r.text[:500]}")
            wait = min(10 * 2 ** i, 300)
            print(f"  {method} {url.replace(base, '')} -> {r.status_code if r is not None else err}; retry in {wait}s")
            time.sleep(wait)
            if hasattr(kw.get("data"), "seek"):
                kw["data"].seek(0)
        raise RuntimeError(f"{method} {url.replace(base, '')} still failing after {tries} tries")

    def md5(p):
        h = hashlib.md5()
        with open(p, "rb") as f:
            for b in iter(lambda: f.read(1 << 20), b""):
                h.update(b)
        return h.hexdigest()

    # 1. Read the draft: new records API first, legacy deposit API as fallback
    api = f"{base}/api/records/{ZENODO_RECORD_ID}/draft"
    try:
        call("GET", api, tries=4)
        entries = call("GET", f"{api}/files", tries=4).json().get("entries", [])
        mode = "rdm"
        for e in [e for e in entries if e.get("status") != "completed"]:
            print("  removing incomplete entry:", e["key"]); call("DELETE", f"{api}/files/{quote(e['key'], safe='')}")
        have = {e["key"]: (e.get("checksum") or "").replace("md5:", "") for e in entries if e.get("status") == "completed"}
    except RuntimeError as e:
        print("Records API failed (", str(e)[:150], ") -> legacy deposit API")
        dep = call("GET", f"{base}/api/deposit/depositions/{ZENODO_RECORD_ID}").json()
        if dep.get("submitted") and dep.get("state") == "done":
            raise RuntimeError("Record is already published: create a new version on the web and use its id.")
        mode, bucket = "legacy", dep["links"]["bucket"]
        have = {f["filename"]: f.get("checksum", "") for f in dep.get("files", [])}
        fid = {f["filename"]: f["id"] for f in dep.get("files", [])}

    def delete(name):
        if mode == "rdm":
            call("DELETE", f"{api}/files/{quote(name, safe='')}")
        else:
            call("DELETE", f"{base}/api/deposit/depositions/{ZENODO_RECORD_ID}/files/{fid[name]}")

    extra = sorted(set(have) - set(upload["file"]))
    print(f"Mode: {mode} | on Zenodo: {len(have)} | planned: {len(upload)} | not in plan: {len(extra)}")
    if extra:
        print("  not in plan:", extra[:8], "..." if len(extra) > 8 else "")
        if DELETE_EXTRA:
            for k in extra:
                delete(k); have.pop(k)
            print("  deleted", len(extra), "files")
    if len(set(have) | set(upload["file"])) > 100:
        raise RuntimeError("จะเกิน 100 ไฟล์ — ตั้ง DELETE_EXTRA = True หรือลบไฟล์ที่ไม่ใช้บนเว็บก่อน")

    # 2. Upload missing or changed files
    uploaded, skipped = [], []
    for _, row in upload.iterrows():
        name, path = str(row["file"]), str(row["path"])
        local = md5(path)
        if have.get(name) == local:
            skipped.append(name); continue
        if name in have:
            print("  replacing changed file:", name); delete(name)
        print(f"Uploading {name} ({row['mb']:.1f} MB)")
        with open(path, "rb") as fp:
            if mode == "rdm":
                fa = f"{api}/files/{quote(name, safe='')}"
                call("POST", f"{api}/files", json=[{"key": name}])
                call("PUT", f"{fa}/content", data=fp, timeout=1800, headers={"Content-Type": "application/octet-stream"})
                remote = (call("POST", f"{fa}/commit").json().get("checksum") or "").replace("md5:", "")
            else:
                remote = (call("PUT", f"{bucket}/{quote(name, safe='')}", data=fp, timeout=1800,
                               headers={"Content-Type": "application/octet-stream"}).json().get("checksum") or "").replace("md5:", "")
        if remote and remote != local:
            raise RuntimeError(f"Checksum mismatch for {name}")
        uploaded.append(name); time.sleep(1)

    print("\n========== ZENODO UPLOAD SUMMARY ==========")
    print("Uploaded now:", len(uploaded), "| skipped (same md5):", len(skipped), "| planned:", len(upload))
    print("Draft URL:", f"{base}/uploads/{ZENODO_RECORD_ID}", "| DOI (reserved):", ZENODO_DOI)
    print("Status: DRAFT — check metadata on the web, then Publish")

Zenodo: https://zenodo.org | record: 23333709
Mode: rdm | on Zenodo: 54 | planned: 64 | not in plan: 1
  not in plan: ['KKN_20240529_labels_15min.nc'] 
Uploading KKN_20240504_cappi_15min.nc (2.4 MB)
Uploading KKN_20240505_cappi_15min.nc (2.9 MB)
Uploading KKN_20240520_cappi_15min.nc (7.3 MB)
Uploading KKN_20240523_cappi_15min.nc (12.5 MB)
Uploading KKN_20240510_polar_15min.nc (20.0 MB)
Uploading KKN_20240516_polar_15min.nc (5.3 MB)
Uploading KKN_20240523_polar_15min.nc (39.8 MB)
Uploading KKN_20240524_polar_15min.nc (16.3 MB)
Uploading KKN_May2024_labels_15min.zip (1.9 MB)
Uploading KKN_May2024_labels_10min_linZ.zip (2.7 MB)
Uploading KKN_May2024_labels_05min_linZ.zip (5.4 MB)

========== ZENODO UPLOAD SUMMARY ==========
Uploaded now: 11 | skipped (same md5): 53 | planned: 64
Draft URL: https://zenodo.org/uploads/23333709 | DOI (reserved): 10.5281/zenodo.23333709
Status: DRAFT — check metadata on the web, then Publish


In [18]:
# สรุปสิ่งที่ต้อง commit เข้า repo
tot = 0
for p in sorted(repo_path("data").rglob("*")):
    if p.is_file() and "cache" not in p.parts:
        tot += p.stat().st_size
print(f"ขนาด data/ ใน repo ทั้งหมด {tot / 1e6:.1f} MB")
for d in ("derived", "days", "sounding", "inventory"):
    q = repo_path("data", d)
    if q.exists():
        print(f"  data/{d}: {sum(f.stat().st_size for f in q.rglob('*') if f.is_file()) / 1e6:.1f} MB")

ขนาด data/ ใน repo ทั้งหมด 108.5 MB
  data/derived: 39.8 MB
  data/days: 59.3 MB
  data/sounding: 1.3 MB
  data/inventory: 2.2 MB


## บันทึกผลเข้า Drive และ (เลือกได้) commit ขึ้น GitHub

`sync.push` เก็บไฟล์ที่จะเข้า repo ไว้ใน `KKN_derived_v1/repo_mirror/` notebook ถัดไป (เครื่องใหม่) จะดึงไปใช้เองตอนเริ่ม **รันเซลล์นี้ทุกครั้งก่อนปิด runtime**

ถ้าต้องการ commit ทันที ตั้ง `COMMIT = True` (ต้องมี secret `KKN_commit` = fine-grained token ของ repo นี้ สิทธิ์ Contents: Read and write) ตั้ง `dry_run=True` เพื่อดูรายการไฟล์ก่อน — ฟังก์ชันไม่พิมพ์ token ออกมา

In [20]:
sync.push(OUT_ROOT)
COMMIT = False ##
if COMMIT:
    sync.git_push("P4: key days, day index, data sources", dry_run=False)

sync.push: /content/Teach_stormTracks_KKN_May2024_HID_doppler → /content/drive/MyDrive/000_research/08Storm_NorthEast_FF69/0data/1radar/KKN_derived_v1/repo_mirror
  คัดลอก 0 ไฟล์ (0.0 MB)
$ git add  (rc=0)
ไฟล์ที่จะ commit:
M	config/data_sources.yaml
M	data/derived/manifest.json
$ git commit  (rc=0)
$ git pull  (rc=0)
$ git push  (rc=0)
✅ push เสร็จ → https://github.com/nattaponm/Teach_stormTracks_KKN_May2024_HID_doppler/tree/main
